# Person 3 — K-Nearest Neighbors & Feature Engineering Pipeline

Pipeline Responsibility: Feature Engineering (RGB/HSV/LBP/HOG)  
Model Assignment: KNN  

This notebook loads the full labelled dataset from `data/raw`, extracts handcrafted features, tunes `k`, and writes results to `outputs/`.


In [ ]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    classification_report, f1_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

MODEL_FOLDER = "knn"
HERE = Path.cwd().resolve()
ROOT = next(
    (
        p
        for p in (HERE, *HERE.parents)
        if (p / "data" / "raw").is_dir() and (p / "Basil_Leaf_ML_Workflow.ipynb").exists()
    ),
    None,
)
if ROOT is None:
    ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find project root containing data/raw.")

MODEL_DIR = ROOT / "parts" / MODEL_FOLDER
OUTPUT_DIR = MODEL_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "parts"))
from _pipeline import CLASSES, FEATURE_VERSION, SEED, prepare_dataset  # noqa: E402

print("Python:", sys.executable)
print("Project root:", ROOT)
print("Model folder:", MODEL_DIR)
print("Outputs:", OUTPUT_DIR)

data = prepare_dataset(ROOT)
X_tr, y_tr = data["X_tr"], data["y_tr"]
X_te, y_te = data["X_te"], data["y_te"]
manifest = data["manifest"]
print(f"Unique images: {len(manifest)} | train: {len(X_tr)} | test: {len(X_te)}")
print("Class counts:", data["audit"]["class_counts"])
print("Dataset complete listed counts:", not data["audit"]["download_coverage"]["partial_dataset"])
print("Feature version:", FEATURE_VERSION)


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

print("--- Person 3: K-Nearest Neighbors ---")
results = []
for k in (1, 3, 5, 7, 9, 11):
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k, weights="distance"))
    pipe.fit(X_tr, y_tr)
    score = f1_score(y_te, pipe.predict(X_te), average="macro", zero_division=0)
    results.append((k, score))
    print(f"k={k}: Macro F1 = {score:.4f}")
best_k = max(results, key=lambda x: x[1])[0]
best_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k, weights="distance"))
start_time = time.perf_counter()
best_knn.fit(X_tr, y_tr)
fit_time = time.perf_counter() - start_time
preds = best_knn.predict(X_te)
acc = accuracy_score(y_te, preds)
p, r, f1, _ = precision_recall_fscore_support(y_te, preds, average="macro", zero_division=0)
cm = confusion_matrix(y_te, preds, labels=CLASSES)
print(f"Optimal k={best_k} | Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")
print("Confusion Matrix:\n", cm)

metrics = {
    "model_name": f"K-Nearest Neighbors (k={best_k})",
    "pipeline_stage": "Feature Engineering",
    "optimal_k": int(best_k),
    "accuracy": float(acc),
    "macro_f1": float(f1),
    "precision": float(p),
    "recall": float(r),
    "fit_time_seconds": float(fit_time),
    "confusion_matrix": cm.tolist(),
    "n_train": int(len(X_tr)),
    "n_test": int(len(X_te)),
    "classes": CLASSES,
    "feature_version": FEATURE_VERSION,
}
(OUTPUT_DIR / "knn_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
joblib.dump(best_knn, OUTPUT_DIR / "knn_model.joblib")
print("Saved outputs to:", OUTPUT_DIR)
